# Bài 2 — Gaussian probabilistic models

**Câu hỏi thực nghiệm:** Giả định độc lập có điều kiện của Gaussian Naive Bayes ảnh hưởng thế nào đến dự đoán? Covariance đầy đủ theo lớp có cải thiện phân loại hoặc ước lượng xác suất không?

Dữ liệu Rice Cammeo/Osmancik gồm 7 đặc trưng số và 2 lớp. Bản ARFF đi kèm có nguồn và trích dẫn trong header. Dùng split phân tầng 80/20 (`SEED=42`), 5-fold CV trên training set và macro-F1 để chọn cấu hình. Test chỉ mở sau khi cấu hình đã chốt.

**Lưu ý khi đọc:** Các nhận xét định lượng được viết theo output đã lưu với Rice ARFF và `SEED=42`. Nếu thay dữ liệu, seed hoặc thư viện, cần đối chiếu lại số liệu.

## Thư viện và cấu hình thực nghiệm

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import arff
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, log_loss

**Cấu hình.** Seed 42 cố định phép chia dữ liệu và các fold. Notebook ưu tiên ARFF cạnh notebook, sau đó tìm từ thư mục gốc repository; nếu thiếu file, thông báo lỗi rõ đường dẫn cần có.

In [2]:
SEED = 42
DATA_PATH = Path("data/Rice_Cammeo_Osmancik.arff")
if not DATA_PATH.exists():
    DATA_PATH = Path("knn-prob-model/data/Rice_Cammeo_Osmancik.arff")
if not DATA_PATH.exists():
    raise FileNotFoundError("Rice ARFF not found beside this notebook or under knn-prob-model/data")

## 1. Đọc và kiểm tra dữ liệu

Xem mẫu dữ liệu, kiểu cột, khoảng biến thiên, giá trị thiếu, hàng trùng và số mẫu từng lớp trước khi huấn luyện. Các cột hình thái là số thực; nhãn `Class` được tách khỏi đặc trưng.

In [3]:
raw, metadata = arff.loadarff(DATA_PATH)

data = pd.DataFrame(raw)

# ARFF nominal labels arrive as bytes; decode them before splitting.
for column in data.select_dtypes(include="object").columns:
    data[column] = data[column].str.decode("utf-8")

target_name = "Class"
X = data.drop(columns=target_name).to_numpy(dtype=float)
y = data[target_name].to_numpy()
feature_names = data.drop(columns=target_name).columns.tolist()
class_names, class_counts = np.unique(y, return_counts=True)

print(f"Dataset: Rice Cammeo/Osmancik | shape={data.shape} | seed={SEED}")
display(data.head())
display(data.dtypes.to_frame("dtype"))
display(data.describe().loc[["min", "max", "std"]].T)
display(pd.DataFrame({"count": class_counts}, index=class_names).rename_axis("class"))
print("Missing:", int(data.isna().sum().sum()), "| duplicates:", int(data.duplicated().sum()))
assert np.isfinite(X).all() and not pd.isna(y).any()

Dataset: Rice Cammeo/Osmancik | shape=(3810, 8) | seed=42


,Area,Perimeter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,Class
0,15231.0,525.578979,229.749878,85.093788,0.928882,15617.0,0.572896,Cammeo
1,14656.0,494.311005,206.020065,91.730972,0.895405,15072.0,0.615436,Cammeo
2,14634.0,501.122009,214.106781,87.768288,0.912118,14954.0,0.693259,Cammeo
3,13176.0,458.342987,193.337387,87.448395,0.891861,13368.0,0.640669,Cammeo
4,14688.0,507.166992,211.743378,89.312454,0.906691,15262.0,0.646024,Cammeo


,dtype
Area,float64
Perimeter,float64
Major_Axis_Length,float64
Minor_Axis_Length,float64
Eccentricity,float64
Convex_Area,float64
Extent,float64
Class,str


,min,max,std
Area,7551.000000,18913.000000,1732.367706
Perimeter,359.100006,548.445984,35.597081
Major_Axis_Length,145.264465,239.010498,17.448679
Minor_Axis_Length,59.532406,107.542450,5.729817
Eccentricity,0.777233,0.948007,0.020818
Convex_Area,7723.000000,19099.000000,1776.972042
Extent,0.497413,0.861050,0.077239


,count
class,
Cammeo,1630
Osmancik,2180


Missing: 0 | duplicates: 0


**Nhận xét dữ liệu.** Rice có 3.810 mẫu, 7 đặc trưng số, 1.630 Cammeo và 2.180 Osmancik. Không có giá trị thiếu hoặc hàng trùng trong ARFF đang dùng. Các cột diện tích có trị số hàng nghìn, còn `Extent` là tỉ lệ; khoảng cách trên dữ liệu gốc vì vậy dễ bị các cột có độ lớn lớn chi phối. Tỉ lệ lớp 42,8%/57,2% là lý do dùng macro-F1 để cho hai lớp trọng số ngang nhau.

### Phân hoạch và giữ test độc lập

Chia phân tầng 80/20 một lần: 3.048 mẫu dùng cho 5-fold CV, 762 mẫu giữ đến cuối. Mỗi fold fit scaler trên chính training fold của nó. Hai notebook dùng cùng split để có thể đối chiếu, nhưng mỗi bài chọn cấu hình riêng.

In [4]:
# Reserve a stratified test set before any model selection.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"Train={len(y_train)}, test={len(y_test)}; test is held out until final evaluation.")

Train=3048, test=762; test is held out until final evaluation.


**Kiểm tra phân hoạch.** Seed, phân tầng và số lượng mẫu được in ra ngay trên. Tập test chưa tham gia bất kỳ phép chọn tham số hay scaler nào ở các phần tiếp theo.

## 2. Gaussian Naive Bayes: giả định và xác suất

Với lớp $c$, prior MLE là $P(c)=n_c/n$; từng đặc trưng có mean $\mu_{cj}$ và phương sai MLE $\sigma^2_{cj}=\frac{1}{n_c}\sum_{i:y_i=c}(x_{ij}-\mu_{cj})^2$. Giả định độc lập có điều kiện cho phép nhân các mật độ một chiều: $P(x\mid c)=\prod_j p(x_j\mid c)$. Tính tổng log mật độ thay cho tích trực tiếp để tránh underflow, rồi chuẩn hóa các log joint score bằng log-sum-exp. Epsilon cộng vào phương sai tránh chia cho 0.

Mỗi phương thức được định nghĩa ở một cell để dễ đọc. Python yêu cầu thân class liền mạch, nên các hàm nhận `self` dưới đây được gắn vào class ở cell cuối.

### `fit`

Ước lượng prior, mean và variance MLE cho từng lớp; cộng smoothing để phương sai không bằng 0.

In [5]:
def _gnb_fit(self, X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    # Estimate priors and per-feature Gaussian parameters on training data.
    self.classes_, counts = np.unique(y, return_counts=True)
    self.class_prior_ = counts / counts.sum()
    self.theta_ = np.vstack([X[y == label].mean(axis=0) for label in self.classes_])
    empirical_var = np.vstack([X[y == label].var(axis=0, ddof=0) for label in self.classes_])

    # Add smoothing to prevent division by zero for constant features.
    self.epsilon_ = self.var_smoothing * max(float(np.var(X, axis=0).max()), np.finfo(float).tiny)
    self.var_ = empirical_var + self.epsilon_
    return self

### `joint_log_likelihood`

Cộng log prior và log likelihood của từng đặc trưng để được điểm chưa chuẩn hóa cho mỗi lớp.

In [6]:
def _gnb_joint_log_likelihood(self, X):
    X = np.asarray(X, dtype=float)
    delta = X[:, None, :] - self.theta_[None, :, :]

    # Sum feature log densities and add the log prior for each class.
    return np.log(self.class_prior_)[None, :] - 0.5 * np.sum(
        np.log(2.0 * np.pi * self.var_)[None, :, :] + delta**2 / self.var_[None, :, :], axis=2
    )

### `predict_log_proba`

Chuẩn hóa các điểm lớp bằng log-sum-exp, tránh underflow khi xác suất rất nhỏ.

In [7]:
def _gnb_predict_log_proba(self, X):
    scores = self.joint_log_likelihood(X)

    # Shift before exponentiating to keep log-sum-exp numerically stable.
    max_score = scores.max(axis=1, keepdims=True)
    log_norm = max_score + np.log(np.exp(scores - max_score).sum(axis=1, keepdims=True))
    return scores - log_norm

### `predict_proba`

Đổi log posterior đã chuẩn hóa về xác suất của từng lớp.

In [8]:
def _gnb_predict_proba(self, X):
    return np.exp(self.predict_log_proba(X))

### `predict`

Chọn lớp có log joint score lớn nhất; phép chuẩn hóa không làm đổi thứ tự lớp.

In [9]:
def _gnb_predict(self, X):
    return self.classes_[np.argmax(self.joint_log_likelihood(X), axis=1)]

### Ghép các phương thức vào `GaussianNBNumPy`

`__init__` lưu cấu hình; các phương thức ở trên được gắn vào class để dùng với `Pipeline` và cross-validation.

In [10]:
class GaussianNBNumPy(ClassifierMixin, BaseEstimator):
    """Gaussian naive Bayes with class-conditional diagonal covariance."""

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    fit = _gnb_fit
    joint_log_likelihood = _gnb_joint_log_likelihood
    predict_log_proba = _gnb_predict_log_proba
    predict_proba = _gnb_predict_proba
    predict = _gnb_predict

### Full covariance theo lớp

Mô hình thứ hai dùng một mean vector và ma trận hiệp phương sai đầy đủ $\Sigma_c$ riêng cho mỗi lớp. Các phần tử ngoài đường chéo có thể mô tả tương quan giữa hai phép đo hình thái. Log-density gồm $\log|\Sigma_c|$ và khoảng cách Mahalanobis $(x-\mu_c)^T\Sigma_c^{-1}(x-\mu_c)$. Ridge $\lambda I$ giúp ma trận khả nghịch; $\lambda$ được chọn trong CV.

Mỗi phương thức được định nghĩa ở một cell để dễ đọc. Python yêu cầu thân class liền mạch, nên các hàm nhận `self` dưới đây được gắn vào class ở cell cuối.

### `fit`

Ước lượng prior, vector mean và ma trận covariance riêng từng lớp; thêm regularization vào đường chéo.

In [11]:
def _full_gaussian_fit(self, X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    # Fit one prior, mean vector and covariance matrix per class.
    self.classes_, counts = np.unique(y, return_counts=True)
    self.class_prior_ = counts / counts.sum()
    self.means_ = np.vstack([X[y == label].mean(axis=0) for label in self.classes_])
    self.covariances_ = []
    for label in self.classes_:
        class_X = X[y == label]
        covariance = np.cov(class_X, rowvar=False, bias=True)

        # Ridge regularization stabilizes inversion for correlated features.
        covariance += self.reg_covar * np.eye(X.shape[1])
        self.covariances_.append(covariance)
    self.covariances_ = np.stack(self.covariances_)
    return self

### `joint_log_likelihood`

Tính log prior cùng log-density Gaussian đa biến từ determinant và khoảng cách Mahalanobis.

In [12]:
def _full_gaussian_joint_log_likelihood(self, X):
    X = np.asarray(X, dtype=float)
    scores = []

    for index, (mean, covariance) in enumerate(zip(self.means_, self.covariances_)):
        sign, logdet = np.linalg.slogdet(covariance)
        if sign <= 0:
            raise np.linalg.LinAlgError("Regularized covariance must be positive definite")

        # Combine Mahalanobis distance, log determinant and class prior.
        delta = X - mean
        quadratic = np.einsum("ni,ij,nj->n", delta, np.linalg.inv(covariance), delta)
        scores.append(np.log(self.class_prior_[index]) - 0.5 * (
            X.shape[1] * np.log(2.0 * np.pi) + logdet + quadratic
        ))
    return np.column_stack(scores)

### `predict_log_proba`

Chuẩn hóa log joint score theo các lớp bằng log-sum-exp để giữ ổn định số.

In [13]:
def _full_gaussian_predict_log_proba(self, X):
    scores = self.joint_log_likelihood(X)

    # Normalize class scores in log space to avoid underflow.
    max_score = scores.max(axis=1, keepdims=True)
    return scores - (max_score + np.log(np.exp(scores - max_score).sum(axis=1, keepdims=True)))

### `predict_proba`

Đưa log posterior về xác suất thông thường cho từng lớp.

In [14]:
def _full_gaussian_predict_proba(self, X):
    return np.exp(self.predict_log_proba(X))

### `predict`

Chọn lớp có log joint score lớn nhất cho mỗi mẫu.

In [15]:
def _full_gaussian_predict(self, X):
    return self.classes_[np.argmax(self.joint_log_likelihood(X), axis=1)]

### Ghép các phương thức vào `GaussianFullCovariance`

`__init__` lưu cấu hình; các phương thức ở trên được gắn vào class để dùng với `Pipeline` và cross-validation.

In [16]:
class GaussianFullCovariance(ClassifierMixin, BaseEstimator):
    """Gaussian discriminant model with a separate covariance per class."""

    def __init__(self, reg_covar=1e-3):
        self.reg_covar = reg_covar

    fit = _full_gaussian_fit
    joint_log_likelihood = _full_gaussian_joint_log_likelihood
    predict_log_proba = _full_gaussian_predict_log_proba
    predict_proba = _full_gaussian_predict_proba
    predict = _full_gaussian_predict

### Kiểm tra mức tương quan trong từng lớp

Nếu các cặp đặc trưng còn tương quan mạnh *sau khi đã cố định lớp*, giả định độc lập có điều kiện của Naive Bayes là một xấp xỉ. Bảng dưới liệt kê các cặp có hệ số tương quan tuyệt đối lớn nhất trên training split; test không tham gia phép khảo sát này.

In [17]:
correlation_rows = []
for label in class_names:
    class_frame = pd.DataFrame(X_train[y_train == label], columns=feature_names)
    correlations = class_frame.corr().abs()
    upper = correlations.where(np.triu(np.ones(correlations.shape, dtype=bool), k=1))
    for (feature_a, feature_b), value in upper.stack().sort_values(ascending=False).head(3).items():
        correlation_rows.append({"class": label, "feature_a": feature_a,
                                 "feature_b": feature_b, "abs_correlation": value})
display(pd.DataFrame(correlation_rows).round(4))

,class,feature_a,feature_b,abs_correlation
0,Cammeo,Area,Convex_Area,0.9974
1,Cammeo,Perimeter,Convex_Area,0.9340
2,Cammeo,Perimeter,Major_Axis_Length,0.9271
3,Osmancik,Area,Convex_Area,0.9978
4,Osmancik,Perimeter,Convex_Area,0.9330
5,Osmancik,Area,Perimeter,0.9270


**Nhận xét bảng tương quan.** `Area` và `Convex_Area` có hệ số tương quan tuyệt đối `0.9974` trong Cammeo và `0.9978` trong Osmancik; `Perimeter` với `Convex_Area` cũng khoảng `0.93`. Các đặc trưng vẫn phụ thuộc mạnh khi đã cố định lớp, nên phép nhân likelihood một chiều của Naive Bayes có thể đếm lặp thông tin. Full covariance được thử để biểu diễn quan hệ ấy; tương quan chưa chứng minh nguyên nhân của chênh lệch điểm số.

## 3. Kiểm tra phép tính và đối chiếu

Ví dụ nhỏ xác nhận prior, mean và tổng xác suất; dữ liệu phương sai bằng 0 kiểm tra ổn định số. Full covariance cũng được thử trên mẫu nhỏ trước khi chạy CV.

In [18]:
toy_X = np.array([[0.0, 0.0], [0.2, 0.1], [2.0, 2.0], [2.2, 1.9]])
toy_y = np.array(["A", "A", "B", "B"])

# Check fitted parameters and normalized probabilities on a small example.
toy_model = GaussianNBNumPy(var_smoothing=1e-9).fit(toy_X, toy_y)
assert np.allclose(toy_model.class_prior_, [0.5, 0.5])
assert np.allclose(toy_model.theta_[0], toy_X[:2].mean(axis=0))
assert np.isfinite(toy_model.predict_proba(toy_X)).all()
assert np.allclose(toy_model.predict_proba(toy_X).sum(axis=1), 1.0)

# Constant features and nearly singular covariance should remain usable.
zero_variance_model = GaussianNBNumPy().fit(
    np.array([[1.0, 0.0], [1.0, 0.0], [3.0, 1.0], [3.0, 1.0]]), toy_y)
assert np.isfinite(zero_variance_model.predict_proba([[1.0, 0.0]])).all()
full_toy = GaussianFullCovariance(reg_covar=1e-3).fit(toy_X, toy_y)
assert np.isfinite(full_toy.predict_proba(toy_X)).all()
assert np.allclose(full_toy.predict_proba(toy_X).sum(axis=1), 1.0)

### Đối chiếu GaussianNB của scikit-learn

Dùng cùng `var_smoothing=1e-9` và fold validation đầu tiên bên trong training split để so xác suất. Tập test vẫn được giữ cho đánh giá cuối.

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Compare compatible smoothing on a validation fold drawn only from train.
reference_train_idx, reference_val_idx = next(cv.split(X_train, y_train))
sk_model = GaussianNB(var_smoothing=1e-9).fit(
    X_train[reference_train_idx], y_train[reference_train_idx])
np_model = GaussianNBNumPy(var_smoothing=1e-9).fit(
    X_train[reference_train_idx], y_train[reference_train_idx])
reference_X = X_train[reference_val_idx]

print("Prediction agreement:",
      np.mean(sk_model.predict(reference_X) == np_model.predict(reference_X)))

print("Maximum probability difference:",
      np.max(np.abs(sk_model.predict_proba(reference_X) - np_model.predict_proba(reference_X))))

assert np.allclose(sk_model.predict_proba(reference_X),
                   np_model.predict_proba(reference_X), atol=1e-8)

Prediction agreement: 1.0
Maximum probability difference: 3.552713678800501e-15


**Nhận xét đối chiếu.** Nhãn dự đoán khớp `100%`, sai khác xác suất lớn nhất `3.55×10⁻¹⁵`. Mức chênh này là sai số số học, cho thấy cách cộng epsilon và chuẩn hóa log posterior phù hợp với cấu hình tham chiếu.

## 4. Chọn cấu hình bằng CV

Mỗi mô hình thử scaling bật/tắt và ba mức smoothing hoặc regularization. `StandardScaler` nằm trong pipeline của từng fold. Macro-F1 là metric chọn cấu hình; log loss được ghi riêng để đánh giá chất lượng xác suất. Khi hòa điểm, ưu tiên độ lệch chuẩn thấp và regularization nhỏ hơn.

In [20]:
rows = []

# Compare the same preprocessing options inside every training fold.
for scale in [False, True]:
    for smoothing in [1e-9, 1e-6, 1e-3]:
        steps = [("scale", StandardScaler())] if scale else []
        steps.append(("model", GaussianNBNumPy(var_smoothing=smoothing)))
        result = cross_validate(Pipeline(steps), X_train, y_train, cv=cv,
            scoring={"macro_f1": "f1_macro", "log_loss": "neg_log_loss"},
            return_train_score=False, n_jobs=1)
        rows.append({"model": "Naive Bayes", "scale": scale, "parameter": smoothing,
                     "cv_macro_f1": result["test_macro_f1"].mean(),
                     "cv_macro_f1_std": result["test_macro_f1"].std(),
                     "cv_log_loss": -result["test_log_loss"].mean()})

    # Full covariance uses its own regularization grid.
    for regularization in [1e-6, 1e-3, 1e-1]:
        steps = [("scale", StandardScaler())] if scale else []
        steps.append(("model", GaussianFullCovariance(reg_covar=regularization)))
        result = cross_validate(Pipeline(steps), X_train, y_train, cv=cv,
            scoring={"macro_f1": "f1_macro", "log_loss": "neg_log_loss"},
            return_train_score=False, n_jobs=1)
        rows.append({"model": "Full covariance", "scale": scale, "parameter": regularization,
                     "cv_macro_f1": result["test_macro_f1"].mean(),
                     "cv_macro_f1_std": result["test_macro_f1"].std(),
                     "cv_log_loss": -result["test_log_loss"].mean()})

# Rank configurations using CV only, before the held-out test is evaluated.

### Bảng CV và quyết định trước test

In [21]:
cv_results = pd.DataFrame(rows).sort_values(
    ["cv_macro_f1", "cv_macro_f1_std", "parameter"],
    ascending=[False, True, True], kind="stable").reset_index(drop=True)
display(cv_results.round(5))
best = cv_results.iloc[0]
best_config = {"model": best.model, "scale": bool(best.scale), "parameter": float(best.parameter)}
print("Selected from CV only:", best_config)

,model,scale,parameter,cv_macro_f1,cv_macro_f1_std,cv_log_loss
0,Full covariance,True,0.100,0.93172,0.00729,0.18262
1,Full covariance,False,0.001,0.93146,0.00970,0.24283
2,Full covariance,False,0.100,0.93072,0.01080,0.21548
3,Full covariance,False,0.000,0.92826,0.01094,0.31184
4,Full covariance,True,0.001,0.92789,0.00800,0.28392
5,Full covariance,True,0.000,0.92696,0.00881,0.34074
6,Naive Bayes,True,0.000,0.92123,0.00912,0.44087
7,Naive Bayes,True,0.000,0.92123,0.00912,0.44087
8,Naive Bayes,True,0.001,0.92123,0.00912,0.43982
9,Naive Bayes,False,0.000,0.91417,0.01240,0.47954


Selected from CV only: {'model': 'Full covariance', 'scale': True, 'parameter': 0.1}


**Nhận xét CV.** Full covariance đã scale với `reg_covar=0.1` có macro-F1 `0.93172` (độ lệch chuẩn `0.00729`), cao hơn Gaussian Naive Bayes tốt nhất `0.92123`. CV log loss tương ứng là `0.18262` và khoảng `0.44087`. Do vậy cấu hình full covariance được chọn làm mô hình chính trước khi mở test; cấu hình Naive Bayes tốt nhất được giữ làm đối chứng.

## 5. Fit cấu hình đã chốt

In [22]:
def make_pipeline(row):
    """Rebuild a selected model with its preprocessing configuration."""
    steps = [("scale", StandardScaler())] if row["scale"] else []
    if row["model"] == "Naive Bayes":
        model = GaussianNBNumPy(var_smoothing=row["parameter"])
    else:
        model = GaussianFullCovariance(reg_covar=row["parameter"])
    steps.append(("model", model))
    return Pipeline(steps)

### Đánh giá test cuối

Mỗi họ mô hình dùng cấu hình tốt nhất của chính nó theo CV. Cả hai được fit lại trên toàn bộ training split rồi báo cáo accuracy, macro-F1, log loss và confusion matrix trên cùng 762 mẫu test.

In [23]:
test_rows = []
fitted_models = {}

# Evaluate the best CV configuration from each model family on the same test set.
for _, row in cv_results.groupby("model", sort=False).head(1).iterrows():
    config = {"model": row.model, "scale": bool(row.scale), "parameter": float(row.parameter)}
    fitted = make_pipeline(config).fit(X_train, y_train)
    pred, prob = fitted.predict(X_test), fitted.predict_proba(X_test)
    test_rows.append({"model": row.model, "configuration": config,
                      "accuracy": accuracy_score(y_test, pred),
                      "macro_f1": f1_score(y_test, pred, average="macro"),
                      "log_loss": log_loss(y_test, prob, labels=fitted.classes_)})
    fitted_models[row.model] = fitted
    display(pd.DataFrame(confusion_matrix(y_test, pred, labels=fitted.classes_),
                         index=fitted.classes_, columns=fitted.classes_).rename_axis("Actual"))

test_results = pd.DataFrame(test_rows).set_index("model")
display(test_results)

,Cammeo,Osmancik
Actual,,
Cammeo,288,38
Osmancik,30,406


,Cammeo,Osmancik
Actual,,
Cammeo,276,50
Osmancik,30,406


,configuration,accuracy,macro_f1,log_loss
model,,,,
Full covariance,"{'model': 'Full covariance', 'scale': True, 'p...",0.910761,0.908569,0.202639
Naive Bayes,"{'model': 'Naive Bayes', 'scale': True, 'param...",0.895013,0.891866,0.554205


**Nhận xét test.** Full covariance đạt accuracy `0.9108`, macro-F1 `0.9086`, log loss `0.2026`; Naive Bayes lần lượt đạt `0.8950`, `0.8919`, `0.5542`. Full covariance nhận đúng thêm 12 Cammeo (288 so với 276), còn số Osmancik đúng bằng nhau (406). Điểm phân loại cao hơn đi cùng log loss thấp hơn ở lần chạy này, cho thấy cả nhãn lẫn xác suất đều tốt hơn theo hai chỉ số đã chọn.

## 6. Giải thích một dự đoán held-out

Mẫu đầu của test được phân tích với mô hình full covariance đã chọn. Likelihood và joint score được trình bày ở miền log; posterior là xác suất sau khi chuẩn hóa giữa hai lớp.

In [24]:
selected_model = fitted_models[best_config["model"]]
model = selected_model.named_steps["model"]
query_index = 0
query = X_test[query_index:query_index + 1]

# Compute each class's prior, likelihood and posterior for the same sample.
query_transformed = selected_model.named_steps["scale"].transform(query) if best_config["scale"] else query
joint = model.joint_log_likelihood(query_transformed)[0]
posterior = model.predict_proba(query_transformed)[0]
explanation = pd.DataFrame({
    "class": model.classes_,
    "prior": model.class_prior_,
    "log_likelihood": joint - np.log(model.class_prior_),
    "log_joint": joint,
    "posterior": posterior,
})
display(explanation)
print("Actual:", y_test[query_index], "| predicted:", selected_model.predict(query)[0])

,class,prior,log_likelihood,log_joint,posterior
0,Cammeo,0.427822,-2.602281,-3.451331,0.991624
1,Osmancik,0.572178,-7.667000,-8.225305,0.008376


Actual: Cammeo | predicted: Cammeo


**Nhận xét mẫu.** Dù prior của Osmancik cao hơn (`0.5722` so với `0.4278`), log-likelihood của mẫu dưới Cammeo cao hơn (`-2.6023` so với `-7.6670`). Vì thế posterior Cammeo đạt `0.9916` và mô hình dự đoán Cammeo, trùng nhãn thật. Ví dụ cho thấy likelihood của đặc trưng có thể đảo ưu thế ban đầu từ prior.

## 7. Thảo luận và giới hạn

Các cặp đặc trưng hình thái có tương quan trong lớp làm giả định độc lập của Naive Bayes khó đúng hoàn toàn; full covariance có thêm tham số để biểu diễn quan hệ đó. Kết quả test ủng hộ full covariance trên Rice trong split này, nhưng không chứng minh tương quan là nguyên nhân duy nhất. Full covariance cũng cần regularization và nhiều mẫu hơn khi số chiều tăng. Một split test và một grid hữu hạn giới hạn khả năng suy rộng của kết luận.